# Sentiment Analysis of Amazon Reviews

This notebook demonstrates how to fine-tune a BERT model for sentiment classification on the Amazon customer reviews dataset.

## 1. Setup

First, let's install the necessary libraries.

In [ ]:
!pip install transformers datasets scikit-learn torch

## 2. Load and Preprocess the Dataset

In [ ]:
from datasets import load_dataset

# Load the dataset
dataset = load_dataset("SetFit/amazon_reviews_multi_en", split='train')

# Map star ratings to sentiment classes
def map_stars_to_sentiment(example):
    if example['stars'] in [1, 2]:
        return {'label': 0}  # negative
    elif example['stars'] == 3:
        return {'label': 1}  # neutral
    else:
        return {'label': 2}  # positive

dataset = dataset.map(map_stars_to_sentiment)

# Split the dataset into training and testing sets (80/20)
dataset = dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = dataset['train']
test_dataset = dataset['test']

print(f"Training set size: {len(train_dataset)}")
print(f"Testing set size: {len(test_dataset)}")

## 3. Tokenization

In [ ]:
from transformers import AutoTokenizer

model_name = 'google-bert/bert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples['review_body'], padding='max_length', truncation=True)

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

# Set format for PyTorch
train_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
test_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

## 4. Model Fine-Tuning

In [ ]:
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments
import numpy as np
from sklearn.metrics import f1_score, accuracy_score

label2id = {'negative': 0, 'neutral': 1, 'positive': 2}
id2label = {0: 'negative', 1: 'neutral', 2: 'positive'}
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=3, 
    id2label=id2label, 
    label2id=label2id
)

def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    f1 = f1_score(labels, preds, average='macro')
    acc = accuracy_score(labels, preds)
    return {'accuracy': acc, 'f1': f1}

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=1,  # For demonstration purposes, we use 1 epoch
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    evaluation_strategy='epoch'
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

## 5. Evaluation

In [ ]:
eval_results = trainer.evaluate()
print(f"Evaluation results: {eval_results}")

## 6. Prediction on New Reviews

In [ ]:
from transformers import pipeline

# Create a prediction pipeline
sentiment_pipeline = pipeline('sentiment-analysis', model=model, tokenizer=tokenizer)

# Sample reviews
new_reviews = [
    "This product is amazing! I love it.",
    "The product is okay, not great but not terrible.",
    "I'm very disappointed with this purchase."
]

# Get predictions
predictions = sentiment_pipeline(new_reviews)

# The pipeline now directly outputs the sentiment label
for review, pred in zip(new_reviews, predictions):
    print(f'Review: "{review}" -> Sentiment: {pred["label"]} (Score: {pred["score"]:.4f})')
